In [ ]:
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path='.env', override=True)

In [2]:
from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [3]:
prompt = """You are a professor and expert in explaining complex topics in a way that is easy to understand. 
Your job is to answer the provided question so that even a 5 year old can understand it. 
You have provided with relevant background context to answer the question.

Question: {question}

Context: {context}

Answer:"""
print(f"Prompt template: {prompt}")

Prompt template: You are a professor and expert in explaining complex topics in a way that is easy to understand. 
Your job is to answer the provided question so that even a 5 year old can understand it. 
You have provided with relevant background context to answer the question.

Question: {question}

Context: {context}

Answer:


In [4]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

@traceable
def search(question):
  web_docs = tavily.search(query=question, max_results=2)

  web_results = "\n".join([d["content"] for d in web_docs["results"]])
  
  return web_results

@traceable
def explain(question, context):
  formatted = prompt.format(question=question, context=context)

  completion = openai_client.chat.completions.create(
    messages=[
      {"role": "system", "content": formatted},
      {"role": "user", "content": question},
    ],
    model="gpt-4o-mini",
  )

  return completion.choices[0].message.content

@traceable
def eli5(question):
  context = search(question)
  answer = explain(question, context)
  return answer

In [5]:
question = "What is the difference between N8N and LangChain?"
print(eli5(question))

Okay! Imagine you have two toys that help you do different things.

**N8N** is like a fun, colorful building block set. You can easily snap the blocks together to make all sorts of cool things, like a simple car or a lovely house. You don’t need to be really good at building or know much about how blocks work; you just connect them! N8N helps you put things together quickly without needing to know a lot about how each piece works.

Now, **LangChain** is more like a complicated puzzle. To finish the puzzle and make something really amazing, you need to understand how each piece fits together. It’s a little harder because you have to put in more effort and think about the whole picture. But once you get good at puzzles, you can create really impressive things with it.

So, the main difference is: N8N is easier and faster for building simple things, while LangChain lets you make more complicated things, but it can take more time and effort to learn.
